# Kiểm tra dữ liệu gốc (tầng 1)

**Mục đích:** biết bộ bình luận crawler ghi ra có những gì trước khi xử lý: video, rỗng, trùng, độ dài, emoji, URL, ngôn ngữ, thời gian, đăng dồn; phân bố theo máy, tuần sau ra mắt, kênh, loại video, reply, like. Notebook chỉ đọc và đếm, không sửa gì. Cập nhật lần cuối: 2026-11-22.

**Nội dung**

1. Đọc dữ liệu
2. Level 0 – Raw Data Audit (số tổng hợp ở cuối mục)
3. Theo máy và thời gian sau ra mắt
4. Theo video, kênh, reply, like
5. Tổng hợp số liệu mục 3–4

**Cách chạy:** chọn kernel Python có pandas, matplotlib, numpy, PyYAML (góc phải trên), rồi *Run All*. Chỉ cần dữ liệu của crawler: `src/ingest/data/raw/comments.jsonl`, `videos.jsonl` và `src/ingest/data/models.json`. Không cần chạy lệnh tầng 2.

**Nhánh `kham-pha-nhan`:** đây là notebook duy nhất trên nhánh. Hướng dẫn chạy: `docs/1-thu-thap/huong-dan-kiem-tra-du-lieu.md`. Số đã chạy: `docs/1-thu-thap/bao-cao-kiem-tra-du-lieu.md`. Bước tiếp theo (khám phá nhãn nội dung): `docs/0-chung/ke-hoach.md`.

In [ ]:
import json
import re
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

from helpers import *   # ROOT, đường dẫn dữ liệu, N_SHOW, doc_jsonl, doc_da_xu_ly, doc_bo_may, lo_ra, bao_cao, display, pd
thiet_lap_hien_thi()

DATA_PATH = RAW_PATH   # chỗ crawler ghi ra
KQ = {}                # con số chính của mục 3–4, in ở mục 5
print("File dữ liệu:", DATA_PATH)

## 1. Đọc dữ liệu

**Để làm gì:** nạp toàn bộ bình luận vào một bảng (DataFrame) để kiểm tra.

**Ô code làm gì:**
- Đọc `comments.jsonl`, mỗi dòng là một bình luận dạng JSON, đúng contract `comment-event` v3.
- Đọc nguyên trạng: `dtype=False`, `convert_dates=False` để pandas không tự đổi kiểu cột (mã, ngày giờ giữ là chuỗi).
- In số dòng, danh sách cột kèm kiểu và số ô trống, rồi 5 bình luận đầu.

**Đọc kết quả:** cột `text` là nội dung bình luận, thứ Level 1 sẽ làm sạch. Ô trống ở `parent_id`, `parent_text` là bình thường: bình luận gốc không có comment cha.

In [ ]:
df = pd.read_json(DATA_PATH, lines=True, dtype=False, convert_dates=False)
print(f"{len(df):,} bình luận × {df.shape[1]} cột")

pd.set_option("display.max_columns", None)    # hiện đủ mọi cột, không ẩn bằng "..."
pd.set_option("display.max_colwidth", None)   # không cắt ngắn bình luận dài
pd.set_option("display.width", None)

# Kiểu và số ô trống của từng trường
display(pd.DataFrame({
    "kiểu": df.dtypes.astype(str),
    "số ô trống (null)": df.isna().sum(),
}))

# Xem dữ liệu theo trang, mỗi trang đủ mọi trường
TRANG = 0        # trang 0 = dòng 0–99, trang 1 = dòng 100–199...; đổi số này rồi chạy lại để lật trang
CO_TRANG = 100   # số dòng mỗi trang
AN_COT = ["author_name"]   # tên kênh người bình luận (dữ liệu cá nhân); đổi thành [] nếu muốn xem
dau, cuoi = TRANG * CO_TRANG, (TRANG + 1) * CO_TRANG
print(f"Dòng {dau:,}–{min(cuoi, len(df)) - 1:,} / {len(df):,}")
display(df.drop(columns=AN_COT, errors="ignore").iloc[dau:cuoi])

## 2. Level 0 – Raw Data Audit

**Để làm gì:** hiểu mình đã crawl được những gì **trước khi sửa bất cứ thứ gì**. Level 0 chỉ đọc và đếm trên cột `text` gốc, không thay đổi dữ liệu.

| Mục | Câu hỏi |
|---|---|
| 2.1 | Crawl được bao nhiêu video, mỗi video bao nhiêu bình luận, mỗi máy bao nhiêu |
| 2.2 | Bao nhiêu bình luận rỗng |
| 2.3 | Bao nhiêu bình luận trùng lặp |
| 2.4 | Bình luận thường dài bao nhiêu |
| 2.5 | Bao nhiêu bình luận chỉ có emoji |
| 2.6 | Bao nhiêu bình luận chỉ có URL |
| 2.7 | Bao nhiêu bình luận tiếng Việt, tiếng Anh, ngôn ngữ khác |
| 2.7b | Trong số bình luận tiếng Việt, bao nhiêu % gõ không dấu |
| 2.8 | Bình luận trải dài bao lâu, có ngày nào tăng đột biến |
| 2.9 | Có nhiều bình luận giống nhau đăng dồn trong thời gian ngắn không |

Cuối mục có bảng tổng hợp các con số chính.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

raw = df["text"].fillna("").astype(str)   # nội dung gốc (null coi như rỗng); chỉ đọc, không sửa
thoi_gian = pd.to_datetime(df["published_at"], utc=True, errors="coerce").dt.tz_convert("Asia/Ho_Chi_Minh")
AUDIT = {}   # con số chính của từng mục, in thành bảng ở cuối Level 0
print(f"{len(df):,} bình luận | thời điểm đăng đọc được: {thoi_gian.notna().sum():,}")

### 2.1 Video và số bình luận mỗi video

**Ô code làm gì:**
- Đếm số video có bình luận, và mỗi video có bao nhiêu bình luận (trung bình, trung vị, các phân vị).
- In `N_SHOW` video nhiều bình luận nhất, kèm tên kênh và tiêu đề lấy từ `videos.jsonl` (nếu có file này).
- Vẽ phân bố số bình luận mỗi video. Trục ngang dùng thang log vì vài video rất đông, còn đa số video ít bình luận.
- Đếm số bình luận và số video theo từng máy (`model`). Video so sánh nhiều máy có `model` dạng `a|b`.

**Đọc kết quả:**
- Trung vị nhỏ hơn trung bình nhiều nghĩa là vài video "hút" phần lớn bình luận.
- Video có quá ít bình luận (dưới 10) thì đóng góp ít cho phân tích.

In [ ]:
VIDEOS_PATH = ROOT / "src" / "ingest" / "data" / "raw" / "videos.jsonl"

cmt_moi_video = df.groupby("video_id").size().sort_values(ascending=False)
AUDIT["Số video có bình luận"] = len(cmt_moi_video)
AUDIT["Video có < 10 bình luận"] = int((cmt_moi_video < 10).sum())
print(f"Số video có bình luận: {len(cmt_moi_video):,} | video có < 10 bình luận: {(cmt_moi_video < 10).sum():,}")
display(cmt_moi_video.describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_frame("bình luận / video"))

vd = None
if VIDEOS_PATH.exists():
    vd = (pd.read_json(VIDEOS_PATH, lines=True, dtype=False, convert_dates=False)
            .drop_duplicates("video_id", keep="last").set_index("video_id"))   # mỗi video lấy bản chụp mới nhất
    khong_cmt = (~vd.index.isin(cmt_moi_video.index)).sum()
    print(f"videos.jsonl có {len(vd):,} video; {khong_cmt:,} video trong đó không có bình luận nào trong comments.jsonl")

top = cmt_moi_video.head(N_SHOW).rename("số bình luận").to_frame()
if vd is not None:
    top = top.join(vd[["channel_title", "title", "published_at"]])
print(f"\n{N_SHOW} video nhiều bình luận nhất:")
display(top)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(cmt_moi_video, bins=np.logspace(0, np.log10(cmt_moi_video.max() + 1), 40), color="#4472c4")
ax.set_xscale("log")
ax.set_xlabel("số bình luận của một video (thang log)")
ax.set_ylabel("số video")
ax.set_title("Phân bố số bình luận mỗi video")
plt.show()

theo_may = (df.groupby("model").agg(so_binh_luan=("comment_id", "size"), so_video=("video_id", "nunique"))
              .sort_values("so_binh_luan", ascending=False))
print(f"\nTheo máy: {len(theo_may):,} giá trị model")
display(theo_may)

### 2.2 Bình luận rỗng

**Ô code làm gì:** đếm bình luận có `text` null, chuỗi rỗng, hoặc chỉ có khoảng trắng, rồi in danh sách.

Bình luận chỉ chứa ký tự vô hình (vd zero-width space) chưa tính ở đây vì nhìn bằng mắt không thấy. Level 1 (B7) sẽ bắt loại này.

In [ ]:
rong = raw.str.strip() == ""
AUDIT["Bình luận rỗng"] = int(rong.sum())
print(f"Bình luận rỗng: {rong.sum():,} / {len(df):,} ({100 * rong.mean():.3f}%)")
display(df.loc[rong, ["comment_id", "video_id", "model", "published_at"]].head(N_SHOW))

### 2.3 Bình luận trùng lặp

**Ô code làm gì:** đếm 3 kiểu trùng.
1. **Trùng `comment_id`:** cùng một bình luận bị ghi hai lần. Đây là lỗi crawl, phải bằng 0.
2. **Trùng nội dung:** khác `comment_id` nhưng nội dung giống hệt nhau (so nguyên văn, bỏ qua bình luận rỗng).
3. **Một người đăng lại cùng nội dung:** cùng `author_hash` và cùng nội dung, xuất hiện từ 2 lần trở lên. Đây là dấu hiệu spam hoặc seeding.

Sau đó in `N_SHOW` nội dung bị lặp nhiều nhất, kèm số lần, số video, số người đăng.

**Đọc kết quả:**
- Câu ngắn phổ biến ("hay", "đẹp quá") trùng nhau là tự nhiên.
- Đáng chú ý là câu **dài** mà trùng nhiều lần, hoặc trùng trên nhiều video.

In [ ]:
trung_id = int(df["comment_id"].duplicated().sum())
co_noi_dung = ~rong
trung_nd = co_noi_dung & raw.duplicated(keep=False)
cung_nguoi = co_noi_dung & df.duplicated(["author_hash", "text"], keep=False)

AUDIT["Trùng comment_id"] = trung_id
AUDIT["Bình luận có nội dung trùng với bình luận khác"] = int(trung_nd.sum())
AUDIT["Bình luận một người đăng lại cùng nội dung"] = int(cung_nguoi.sum())
print(f"Trùng comment_id: {trung_id:,}")
print(f"Bình luận có nội dung trùng với bình luận khác: {trung_nd.sum():,} ({100 * trung_nd.mean():.2f}%), "
      f"gồm {raw[trung_nd].nunique():,} nội dung khác nhau")
print(f"Bình luận mà cùng một người đăng lại cùng nội dung: {cung_nguoi.sum():,}")

lap = (df[trung_nd].assign(text=raw[trung_nd])
         .groupby("text").agg(so_lan=("comment_id", "size"), so_video=("video_id", "nunique"),
                              so_nguoi=("author_hash", "nunique"))
         .sort_values("so_lan", ascending=False))
lap.insert(0, "do_dai", lap.index.str.len())
print(f"\n{N_SHOW} nội dung lặp nhiều nhất:")
display(lap.head(N_SHOW))
print(f"\n{N_SHOW} nội dung DÀI (≥ 30 ký tự) lặp nhiều nhất:")
display(lap[lap["do_dai"] >= 30].head(N_SHOW))

### 2.4 Độ dài bình luận

**Ô code làm gì:**
- Đo độ dài theo **số ký tự** và **số từ** (từ = cụm ngăn bởi khoảng trắng), bỏ qua bình luận rỗng.
- In trung bình, trung vị và các phân vị.
- Vẽ phân bố, cắt ở phân vị 99% để vài bình luận cực dài không làm bẹp biểu đồ.
- Đếm bình luận rất ngắn (≤ 2 từ) và rất dài (> 1.000 ký tự).

**Đọc kết quả:**
- Trung vị là độ dài "thường gặp".
- Bình luận rất ngắn thường không đủ ý để gán khía cạnh và cảm xúc.

In [ ]:
co_chu = raw[~rong]
so_ky_tu = co_chu.str.len()
so_tu = co_chu.str.split().str.len()
display(pd.DataFrame({"số ký tự": so_ky_tu.describe(percentiles=[.1, .25, .5, .75, .9, .99]),
                      "số từ": so_tu.describe(percentiles=[.1, .25, .5, .75, .9, .99])}).round(1))
AUDIT["Độ dài trung vị (ký tự)"] = float(so_ky_tu.median())
AUDIT["Độ dài trung vị (từ)"] = float(so_tu.median())
AUDIT["Bình luận ≤ 2 từ"] = int((so_tu <= 2).sum())
AUDIT["Bình luận > 1.000 ký tự"] = int((so_ky_tu > 1000).sum())
print(f"≤ 2 từ: {(so_tu <= 2).sum():,} | > 1.000 ký tự: {(so_ky_tu > 1000).sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, s, ten in [(axes[0], so_ky_tu, "số ký tự"), (axes[1], so_tu, "số từ")]:
    ax.hist(s.clip(upper=s.quantile(.99)), bins=50, color="#4472c4")
    ax.axvline(s.median(), color="#c00000", ls="--", label=f"trung vị = {s.median():.0f}")
    ax.set_xlabel(f"{ten} (cắt ở phân vị 99%)")
    ax.set_ylabel("số bình luận")
    ax.legend()
fig.suptitle("Độ dài bình luận")
plt.show()

### 2.5 Bình luận chỉ có emoji

**Ô code làm gì:**
- Một bình luận "chỉ có emoji" khi mọi ký tự (trừ khoảng trắng) là emoji hoặc biểu tượng: 😂, ❤️, 👍🏻, 👨‍👩‍👧, ★…
- Ký tự nối emoji (ZWJ, `U+FE0F`, màu da) được tính là một phần của emoji.
- Có một chữ cái hay chữ số là không tính.
- In số lượng, các bình luận emoji gặp nhiều nhất, và một số ví dụ.

**Đọc kết quả:** loại này mang cảm xúc nhưng không có khía cạnh, nên không dùng được cho Q2–Q5.

In [ ]:
CO_CHU_HOAC_SO = re.compile(r"[^\W_]")   # chữ cái hoặc chữ số, ở mọi ngôn ngữ


def _la_ky_tu_emoji(ch: str) -> bool:
    cp = ord(ch)
    return (unicodedata.category(ch) in ("So", "Sk") or 0x1F000 <= cp <= 0x1FAFF or 0x2600 <= cp <= 0x27BF
            or ch in "‍️⃣" or 0xE0020 <= cp <= 0xE007F)


def chi_co_emoji(s: str) -> bool:
    if CO_CHU_HOAC_SO.search(s):            # có chữ/số thì loại ngay, cho nhanh
        return False
    ky_tu = [ch for ch in s if not ch.isspace()]
    return bool(ky_tu) and all(_la_ky_tu_emoji(ch) for ch in ky_tu)


emoji_only = raw.map(chi_co_emoji)
AUDIT["Bình luận chỉ có emoji"] = int(emoji_only.sum())
print(f"Bình luận chỉ có emoji: {emoji_only.sum():,} ({100 * emoji_only.mean():.2f}%)")
display(raw[emoji_only].value_counts().head(N_SHOW).rename("số lần").to_frame())

### 2.6 Bình luận chỉ có URL

**Ô code làm gì:**
- Tìm link trong bình luận: `http://…`, `https://…`, `www.…`, hoặc dạng `tên.com`, `tên.vn`, `youtu.be/…`.
- Một bình luận "chỉ có URL" khi bỏ hết link đi thì không còn chữ cái hay chữ số nào.
- Đếm thêm số bình luận **có chứa** URL để so sánh.

**Đọc kết quả:** bình luận chỉ có URL gần như chắc chắn là quảng cáo hoặc spam.

In [ ]:
URL_RE = re.compile(r"(?:https?://|www\.)\S+"
                    r"|\b[\w-]+(?:\.[\w-]+)*\.(?:com|vn|net|org|info|io|me|ly|gl|be|co|xyz|shop|top|site|link)\b(?:/\S*)?",
                    re.IGNORECASE)


def chi_co_url(s: str) -> bool:
    if not URL_RE.search(s):
        return False
    return not CO_CHU_HOAC_SO.search(URL_RE.sub(" ", s))


co_url = raw.str.contains(URL_RE)
url_only = raw.map(chi_co_url)
AUDIT["Bình luận có chứa URL"] = int(co_url.sum())
AUDIT["Bình luận chỉ có URL"] = int(url_only.sum())
print(f"Có chứa URL: {co_url.sum():,} ({100 * co_url.mean():.2f}%) | chỉ có URL: {url_only.sum():,}")
display(df.loc[url_only, ["comment_id", "video_id", "model", "text"]].head(N_SHOW))

### 2.7 Ngôn ngữ

**Ô code làm gì:** đoán ngôn ngữ của từng bình luận bằng luật đơn giản, xét theo thứ tự:

| Nhãn | Khi nào |
|---|---|
| tiếng Việt | Có chữ cái tiếng Việt có dấu (ă, â, đ, ê, ô, ơ, ư, hoặc nguyên âm mang dấu thanh) |
| ngôn ngữ khác | Có chữ viết không phải chữ Latin: Trung, Nhật, Hàn, Thái, Nga, Ả Rập… |
| tiếng Việt không dấu | Chữ Latin không dấu, có nhiều từ tiếng Việt thường gặp ("ko", "dep", "nhung"…) hơn từ tiếng Anh |
| tiếng Anh | Chữ Latin không dấu, có nhiều từ tiếng Anh thường gặp ("the", "is", "this"…) hơn từ tiếng Việt |
| không rõ | Chữ Latin không dấu nhưng không khớp danh sách nào, vd chỉ có tên máy "iphone 15 pro max" |
| không có chữ | Chỉ có emoji, số, ký hiệu, hoặc rỗng |

- Trước khi xét, bình luận được chuẩn NFC, chỉ để nhận dấu cho đúng. Cột `text` không bị sửa.
- Đây là **ước lượng**: chưa dùng mô hình nhận dạng ngôn ngữ. Ô code in ví dụ ngẫu nhiên cho từng nhãn để kiểm tra bằng mắt.

In [ ]:
CHU_VIET_CO_DAU = set("ăâđêôơưàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵ")
TU_VIET = {"ko", "k", "khong", "kh", "la", "cua", "may", "dep", "qua", "nhung", "thi", "co", "cho", "nay", "duoc", "dc",
           "roi", "minh", "toi", "em", "anh", "ae", "gi", "sao", "nhu", "lam", "hon", "van", "chi", "con", "nua", "voi",
           "mua", "dung", "xai", "gia", "tot", "xau", "ngon", "re", "hay", "cung", "vay", "nhi", "nhe", "oi", "ak", "ah",
           "j", "r", "ms", "mik", "mn", "bn", "vs", "mau", "nong", "lag", "ngu", "chup", "sac", "trau", "yeu", "manh"}
TU_ANH = {"the", "is", "are", "and", "of", "it", "this", "that", "you", "my", "for", "with", "not", "but", "have",
          "was", "what", "just", "like", "its", "be", "at", "your", "they", "will", "can", "would", "should", "than",
          "from", "about", "if", "or", "has", "had", "been", "very", "really", "better", "best", "love", "nice",
          "great", "awesome", "please", "thanks", "who", "why", "how", "when", "which", "there", "their"}
TU_CHU = re.compile(r"[^\W\d_]+")


def _la_chu_latin(ch: str) -> bool:
    cp = ord(ch)
    return cp < 0x250 or 0x1E00 <= cp <= 0x1EFF


def doan_ngon_ngu(s: str) -> str:
    s = unicodedata.normalize("NFC", s).lower()
    tu = TU_CHU.findall(s)
    if not tu:
        return "không có chữ"
    if any(ch in CHU_VIET_CO_DAU for ch in s):
        return "tiếng Việt"
    if any(not _la_chu_latin(ch) for t in tu for ch in t):
        return "ngôn ngữ khác"
    viet = sum(t in TU_VIET for t in tu)
    anh = sum(t in TU_ANH for t in tu)
    if viet > anh:
        return "tiếng Việt không dấu"
    if anh > viet:
        return "tiếng Anh"
    return "không rõ"


ngon_ngu = raw.map(doan_ngon_ngu)
bang_nn = ngon_ngu.value_counts().rename("số bình luận").to_frame()
bang_nn["tỷ lệ (%)"] = (100 * bang_nn["số bình luận"] / len(df)).round(2)
display(bang_nn)
for nhan, so in ngon_ngu.value_counts().items():
    AUDIT[f"Ngôn ngữ: {nhan}"] = int(so)

for nhan in bang_nn.index:
    if nhan == "không có chữ":
        continue
    mau = raw[ngon_ngu == nhan]
    print(f"\nVí dụ ngẫu nhiên – {nhan}:")
    display(mau.sample(min(10, len(mau)), random_state=0).to_frame("text"))

### 2.7b Gõ không dấu

**Ô code làm gì:** dùng nhãn ngôn ngữ của mục 2.7, tính trong số bình luận tiếng Việt có bao nhiêu % gõ không dấu.

**Đọc kết quả:** Bước chuẩn hóa (2.2) **không thêm dấu** cho chữ không dấu ("pin trau qua" vẫn giữ nguyên). Vì vậy từ khóa, từ điển và mô hình ở bước sau phải xử lý được cả hai dạng; tỷ lệ này cho biết việc đó quan trọng tới đâu.

In [ ]:
la_viet = ngon_ngu.isin(["tiếng Việt", "tiếng Việt không dấu"])
khong_dau = ngon_ngu == "tiếng Việt không dấu"
ty_le_kd = round(100 * khong_dau.sum() / max(int(la_viet.sum()), 1), 2)
AUDIT["% tiếng Việt gõ không dấu (trong số tiếng Việt)"] = ty_le_kd
print(f"Tiếng Việt: {la_viet.sum():,} bình luận, trong đó gõ không dấu: {khong_dau.sum():,} ({ty_le_kd:.2f}%)")

### 2.8 Thời gian: trải dài bao lâu, ngày nào tăng đột biến

**Ô code làm gì:**
- In bình luận sớm nhất, muộn nhất, khoảng thời gian trải dài. Giờ đổi sang giờ Việt Nam.
- Đếm số bình luận mỗi ngày và vẽ biểu đồ theo thời gian.
- Đánh dấu **ngày đột biến**: ngày có số bình luận ≥ `HE_SO_DOT_BIEN` lần trung vị của 7 ngày quanh nó, và ≥ `TOI_THIEU_NGAY` bình luận (để bỏ qua ngày lặng).
- Với mỗi ngày đột biến, in video và máy đóng góp nhiều nhất hôm đó, để biết vì sao tăng.

**Đọc kết quả:**
- Crawler chỉ lấy bình luận trong 4 tuần sau ngày ra mắt của từng máy. Vì vậy biểu đồ thành từng cụm quanh các đợt ra mắt, giữa các cụm có khoảng trống. Đó là do cách crawl, không phải người xem bỏ đi.
- Ngày đột biến thường do một video "nổ".

In [ ]:
HE_SO_DOT_BIEN = 3      # gấp mấy lần trung vị 7 ngày quanh đó thì tính là đột biến
TOI_THIEU_NGAY = 50     # ngày ít hơn số này thì không xét

t = thoi_gian.dropna()
print(f"Sớm nhất: {t.min()} | muộn nhất: {t.max()} | trải dài {(t.max() - t.min()).days:,} ngày")
print(f"Bình luận không đọc được thời điểm đăng: {thoi_gian.isna().sum():,}")
AUDIT["Ngày sớm nhất"] = str(t.min().date())
AUDIT["Ngày muộn nhất"] = str(t.max().date())

theo_ngay = t.dt.floor("D").value_counts().sort_index()
theo_ngay = theo_ngay.asfreq("D", fill_value=0)   # thêm cả ngày không có bình luận (= 0)
nen = theo_ngay.rolling(7, center=True, min_periods=1).median()
dot_bien = theo_ngay[(theo_ngay >= HE_SO_DOT_BIEN * nen.clip(lower=1)) & (theo_ngay >= TOI_THIEU_NGAY)]
AUDIT["Số ngày đột biến"] = len(dot_bien)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(theo_ngay.index, theo_ngay.values, lw=0.7, color="#4472c4", label="bình luận / ngày")
ax.scatter(dot_bien.index, dot_bien.values, color="#c00000", s=12, zorder=3, label="ngày đột biến")
ax.set_ylabel("số bình luận")
ax.set_title("Số bình luận mỗi ngày")
ax.legend()
plt.show()

ngay = thoi_gian.dt.floor("D")
chi_tiet = []
for d, so in dot_bien.sort_values(ascending=False).items():
    hom_do = df[ngay == d]
    v = hom_do["video_id"].value_counts()
    chi_tiet.append({"ngày": d.date(), "số bình luận": so, "trung vị 7 ngày": nen[d],
                     "máy nhiều nhất": hom_do["model"].value_counts().index[0],
                     "video nhiều nhất": v.index[0], "bình luận của video đó": v.iloc[0],
                     "tiêu đề": vd["title"].get(v.index[0], "") if vd is not None else ""})
print(f"Ngày đột biến: {len(dot_bien):,}")
display(pd.DataFrame(chi_tiet).head(N_SHOW))

### 2.9 Nhiều bình luận giống nhau đăng dồn trong thời gian ngắn

**Để làm gì:** nhiều bình luận cùng nội dung đăng dồn trong vài phút là dấu hiệu bot, seeding, hoặc spam hàng loạt.

**Ô code làm gì:**
- So nội dung sau khi bỏ khác biệt hoa/thường và khoảng trắng.
- Chỉ xét nội dung dài ≥ `TOI_THIEU_KY_TU` ký tự, để bỏ qua câu ngắn tự nhiên như "hay".
- Một nhóm bị đánh dấu khi có ≥ `TOI_THIEU_LAN` bình luận cùng nội dung nằm trong một khung `CUA_SO`.
- In các nhóm đó, kèm:
  - số lần đăng trong khung dày nhất;
  - thời điểm bắt đầu khung;
  - số video, số người đăng.

**Đọc kết quả:**
- Nhiều người, nhiều video, cùng nội dung, trong vài phút: khả năng cao là chiến dịch.
- Một người đăng lặp: spam cá nhân.

In [ ]:
CUA_SO = pd.Timedelta(minutes=10)
TOI_THIEU_LAN = 3
TOI_THIEU_KY_TU = 10

khoa = raw.str.lower().str.split().str.join(" ")   # bỏ khác biệt hoa/thường và khoảng trắng
giay = (thoi_gian - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds()
xet = (khoa.str.len() >= TOI_THIEU_KY_TU) & khoa.duplicated(keep=False) & giay.notna()
d = pd.DataFrame({"khoa": khoa[xet], "giay": giay[xet], "video_id": df.loc[xet, "video_id"],
                  "author_hash": df.loc[xet, "author_hash"]})

nhom = []
w = CUA_SO.total_seconds()
for k, g in d.groupby("khoa", sort=False):
    if len(g) < TOI_THIEU_LAN:
        continue
    g = g.sort_values("giay")
    s = g["giay"].to_numpy()
    dem = np.searchsorted(s, s + w, side="right") - np.arange(len(s))   # số bình luận trong khung bắt đầu ở mỗi lần đăng
    i = int(dem.argmax())
    if dem[i] >= TOI_THIEU_LAN:
        khung = g.iloc[i:i + dem[i]]
        nhom.append({"nội dung": k, "tổng số lần": len(g), "số lần trong khung": int(dem[i]),
                     "bắt đầu": pd.Timestamp(s[i], unit="s", tz="UTC").tz_convert("Asia/Ho_Chi_Minh"),
                     "số video": khung["video_id"].nunique(), "số người": khung["author_hash"].nunique()})

dang_don = pd.DataFrame(nhom)
if len(dang_don):
    dang_don = dang_don.sort_values(["số lần trong khung", "tổng số lần"], ascending=False)
AUDIT["Nhóm nội dung đăng dồn"] = len(dang_don)
AUDIT["Bình luận trong các khung đăng dồn"] = int(dang_don["số lần trong khung"].sum()) if len(dang_don) else 0
print(f"Nhóm nội dung có ≥ {TOI_THIEU_LAN} lần đăng trong {CUA_SO}: {len(dang_don):,}")
display(dang_don.head(N_SHOW))

### Tổng hợp Level 0

**Đọc kết quả:** các con số chính của mục 2.1–2.9 trên một bảng, để chép vào báo cáo hay so giữa các lần crawl.

In [ ]:
display(pd.Series(AUDIT, name="giá trị").to_frame())

## 3. Theo máy và thời gian sau ra mắt

**Để làm gì:** biết máy nào đủ dữ liệu để phân tích, và bình luận rải qua 4 tuần sau ra mắt ra sao. Đây là đầu vào để chốt bộ 4 máy (chưa chốt) và để biết tuần nào ít dữ liệu.

**Khác mục 2.1:** mục 2.1 đếm theo giá trị `model` thô (video so sánh có `model` dạng `a|b`). Mục này tách `a|b` thành từng máy, nên một bình luận dưới video so sánh được tính cho cả hai máy (gọi là "lượt"). Ngày ra mắt lấy từ `src/ingest/data/models.json`, file crawler sinh ra từ `config/pipeline.yaml`.

**Cách tính tuần:** ngày 0 là ngày ra mắt, tính từ 00:00 giờ Việt Nam (`window_start` trong `models.json`). Tuần 1 = ngày 0–6, tuần 2 = ngày 7–13… Bình luận nằm ngoài tuần 1–4 được đếm riêng ở cột "ngoài cửa sổ".

**Lưu ý:** đây là máy **của video**, chưa phải máy được nhắc trong bình luận (muốn biết phải đọc nội dung, xem kế hoạch nhánh). Một bình luận dưới video S26 Ultra có thể đang nói về iPhone.

### 3.1 Số bình luận theo từng máy

**Ô code làm gì:**
- Mỗi máy: tên, hãng, ngày ra mắt, số lượt bình luận, số video, % là reply, số bình luận từng tuần 1–4 và ngoài cửa sổ.
- Đánh dấu ⚠ máy có ít hơn `NGUONG_BINH_LUAN` lượt. Ngưỡng này chỉ để lọc thô; muốn biết đủ dữ liệu theo từng khía cạnh hay không thì phải có nhãn nội dung (kế hoạch nhánh).
- In máy có trong config nhưng không có bình luận nào, và bảng gộp theo hãng.
- Hãng suy từ chữ đầu của tên máy (`iPhone` → Apple, `Redmi` → Xiaomi…).

In [ ]:
# MODELS_PATH (src/ingest/data/models.json, crawler sinh ra từ config/pipeline.yaml) lấy từ helpers
NGUONG_BINH_LUAN = 3000   # máy ít hơn số này bị đánh dấu ⚠ (ngưỡng tạm)
HANG = {"iphone": "Apple", "samsung": "Samsung", "xiaomi": "Xiaomi", "redmi": "Xiaomi", "vivo": "Vivo", "oppo": "Oppo",
        "google": "Google", "huawei": "Huawei", "realme": "Realme", "honor": "Honor", "oneplus": "OnePlus",
        "sony": "Sony", "lg": "LG", "htc": "HTC"}

danh_muc = pd.DataFrame(json.loads(MODELS_PATH.read_text(encoding="utf-8"))["models"]).set_index("name")
danh_muc["window_start"] = pd.to_datetime(danh_muc["window_start"], utc=True)
danh_muc["hãng"] = danh_muc["title"].str.split().str[0].str.lower().map(HANG).fillna("khác")

# Mỗi bình luận một dòng cho mỗi máy của video: video so sánh "a|b" cho ra 2 dòng (2 lượt)
bl_may = pd.DataFrame({
    "comment_id": df["comment_id"], "video_id": df["video_id"], "la_reply": df["parent_id"].notna(),
    "may": df["model"].fillna("").str.split("|"), "thoi_gian": thoi_gian,
}).explode("may")
bl_may = bl_may[bl_may["may"] != ""]
ngay_sau = (bl_may["thoi_gian"] - bl_may["may"].map(danh_muc["window_start"])).dt.total_seconds() / 86400
bl_may["ngay"] = np.floor(ngay_sau)           # ngày 0 = ngày ra mắt
bl_may["tuan"] = np.floor(ngay_sau / 7) + 1   # tuần 1 = ngày 0–6
trong_cua_so = bl_may["tuan"].between(1, 4)

bang_may = bl_may.groupby("may").agg(**{"bình luận": ("comment_id", "size"), "video": ("video_id", "nunique"),
                                        "% reply": ("la_reply", "mean")})
bang_may["% reply"] = (100 * bang_may["% reply"]).round(1)
tuan = (bl_may[trong_cua_so].groupby(["may", "tuan"]).size().unstack(fill_value=0)
          .reindex(columns=[1.0, 2.0, 3.0, 4.0], fill_value=0))
tuan.columns = [f"tuần {int(c)}" for c in tuan.columns]
bang_may = (danh_muc[["title", "hãng", "launch_date"]].rename(columns={"title": "tên", "launch_date": "ra mắt"})
              .join(bang_may, how="right").join(tuan))
bang_may["ngoài cửa sổ"] = bl_may[~trong_cua_so].groupby("may").size()
COT_DEM = [*tuan.columns, "ngoài cửa sổ"]
bang_may[COT_DEM] = bang_may[COT_DEM].fillna(0).astype(int)
bang_may["ít dữ liệu"] = np.where(bang_may["bình luận"] < NGUONG_BINH_LUAN, "⚠", "")
bang_may = bang_may.sort_values("bình luận", ascending=False)

khong_co = sorted(danh_muc.index.difference(bang_may.index))
KQ["Số máy có bình luận"] = len(bang_may)
KQ[f"Máy có < {NGUONG_BINH_LUAN:,} lượt bình luận"] = int((bang_may["bình luận"] < NGUONG_BINH_LUAN).sum())
KQ["Máy khai trong config nhưng không có bình luận"] = len(khong_co)
KQ["Lượt bình luận ngoài tuần 1–4"] = int((~trong_cua_so).sum())
print(f"{len(bang_may):,} máy có bình luận | {len(bl_may):,} lượt (một bình luận dưới video so sánh 2 máy = 2 lượt)")
print(f"Máy có < {NGUONG_BINH_LUAN:,} lượt: {(bang_may['bình luận'] < NGUONG_BINH_LUAN).sum():,} | lượt ngoài tuần 1–4: {(~trong_cua_so).sum():,}")
display(bang_may)
if khong_co:
    print(f"Máy có trong config nhưng không có bình luận ({len(khong_co)}):", ", ".join(khong_co))

theo_hang = (bang_may.groupby("hãng").agg(**{"số máy": ("bình luận", "size"), "lượt bình luận": ("bình luận", "sum")})
               .sort_values("lượt bình luận", ascending=False))
theo_hang["% lượt"] = (100 * theo_hang["lượt bình luận"] / theo_hang["lượt bình luận"].sum()).round(1)
display(theo_hang)

### 3.2 Bình luận theo ngày sau ra mắt (gộp mọi máy)

**Ô code làm gì:** đếm lượt bình luận theo ngày 0–27 sau ra mắt, gộp mọi máy; in tỷ lệ theo tuần 1–4.

**Đọc kết quả:** bình luận thường dồn vào mấy ngày đầu rồi giảm dần. Tuần 3–4 càng ít thì so sánh "nửa đầu – nửa sau" (Q5) và đường theo tuần (Q1) càng kém chắc.

In [ ]:
theo_ngay_sau = (bl_may.loc[bl_may["ngay"].between(0, 27), "ngay"].astype(int)
                   .value_counts().reindex(range(28), fill_value=0))
ty_le_tuan = bl_may.loc[trong_cua_so, "tuan"].value_counts(normalize=True).sort_index()
for k, v in ty_le_tuan.items():
    KQ[f"% lượt bình luận tuần {int(k)}"] = round(100 * v, 1)
print("Tỷ lệ theo tuần:", " | ".join(f"tuần {int(k)}: {100 * v:.1f}%" for k, v in ty_le_tuan.items()))

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(theo_ngay_sau.index, theo_ngay_sau.values, color="#4472c4")
for x in (7, 14, 21):
    ax.axvline(x - 0.5, color="#999999", lw=0.8, ls="--")
ax.set_xlabel("ngày sau ra mắt (0 = ngày ra mắt)")
ax.set_ylabel("lượt bình luận")
ax.set_title("Bình luận theo ngày sau ra mắt, gộp mọi máy")
plt.show()

### 3.3 Theo ngày, từng máy

**Ô code làm gì:** vẽ số bình luận theo ngày 0–27 cho các máy trong `MAY_XEM`. Đổi danh sách để xem máy khác; tên máy lấy ở cột đầu bảng 3.1.

In [ ]:
MAY_XEM = ["s26u", "s25u", "ip17prm", "xiaomi17u"]   # đổi để xem máy khác

fig, ax = plt.subplots(figsize=(10, 3.5))
for m in MAY_XEM:
    s = (bl_may.loc[(bl_may["may"] == m) & bl_may["ngay"].between(0, 27), "ngay"].astype(int)
           .value_counts().reindex(range(28), fill_value=0))
    ax.plot(s.index, s.values, marker=".", label=f"{m} ({s.sum():,})")
for x in (7, 14, 21):
    ax.axvline(x - 0.5, color="#999999", lw=0.8, ls="--")
ax.set_xlabel("ngày sau ra mắt")
ax.set_ylabel("lượt bình luận")
ax.set_title("Bình luận theo ngày, từng máy")
ax.legend()
plt.show()

### 3.4 Bộ 4 máy ứng viên

**Ô code làm gì:** in số bình luận, số video, số bình luận từng tuần của bộ 4 máy (X, đời trước P, hai đối thủ R1, R2).

**Bộ máy lấy từ `config/analysis.yaml`.** Bộ chính thức chưa chốt: file đó là bộ cũ, chỉ để xem thử. Muốn thử bộ khác thì gán lại `BO_MAY` trong ô, vd `{"X": "s26u", "P": "s25u", "R1": "ip17prm", "R2": "xiaomi17u"}`.

Bộ 4 máy có đủ dữ liệu cho từng khía cạnh không: phải có nhãn nội dung mới biết (kế hoạch nhánh).

In [ ]:
BO_MAY = doc_bo_may()   # từ config/analysis.yaml
nhan = [f"{vai} = {m}" for vai, m in BO_MAY.items()]
tom_tat = bang_may.reindex(list(BO_MAY.values()))[["tên", "ra mắt", "bình luận", "video", *tuan.columns]]
tom_tat.index = nhan
thieu = [m for m in BO_MAY.values() if m not in bang_may.index]
if thieu:
    print("Máy không có bình luận nào:", ", ".join(thieu))
display(tom_tat)

## 4. Theo video, kênh, reply, like

**Để làm gì:** biết dữ liệu dựa vào bao nhiêu kênh, loại video nào, có bao nhiêu video tặng quà kéo bình luận rác vào, và cấu trúc hỏi–đáp (reply) ra sao.

Thông tin video (kênh, tiêu đề, loại) lấy từ `videos.jsonl` đã đọc ở mục 2.1 (biến `vd`). Không có file đó thì mục 4.1–4.3 bỏ qua.

### 4.1 Theo kênh

**Ô code làm gì:** số bình luận, số video của từng kênh, % trên tổng số bình luận và % cộng dồn từ kênh lớn nhất.

**Đọc kết quả:** vài kênh chiếm phần lớn bình luận thì kết quả nghiêng theo khán giả của các kênh đó. Khi phân tích nên xem kết quả có giữ được khi bỏ kênh lớn nhất ra không.

In [ ]:
if vd is None:
    print("Không có videos.jsonl: bỏ qua mục 4.1–4.3")
else:
    bl_vd = df[["comment_id", "video_id"]].join(vd.reindex(columns=["channel_title", "video_type", "title"]), on="video_id")
    bl_vd["channel_title"] = bl_vd["channel_title"].fillna("(video không có trong videos.jsonl)")
    theo_kenh = (bl_vd.groupby("channel_title").agg(**{"bình luận": ("comment_id", "size"), "video": ("video_id", "nunique")})
                   .sort_values("bình luận", ascending=False))
    theo_kenh["% bình luận"] = (100 * theo_kenh["bình luận"] / len(df)).round(2)
    theo_kenh["% cộng dồn"] = theo_kenh["% bình luận"].cumsum().round(1)
    KQ["Số kênh có bình luận"] = len(theo_kenh)
    KQ["% bình luận thuộc 5 kênh lớn nhất"] = round(float(theo_kenh["% bình luận"].head(5).sum()), 1)
    KQ["% bình luận thuộc 10 kênh lớn nhất"] = round(float(theo_kenh["% bình luận"].head(10).sum()), 1)
    print(f"{len(theo_kenh):,} kênh | 5 kênh lớn nhất: {theo_kenh['% bình luận'].head(5).sum():.1f}% bình luận"
          f" | 10 kênh: {theo_kenh['% bình luận'].head(10).sum():.1f}%")
    display(theo_kenh.head(N_SHOW))

### 4.2 Theo loại video

**Ô code làm gì:** số video, số bình luận theo `video_type` crawler gán từ tiêu đề: `review`, `unbox` (đập hộp, trên tay), `compare` (so sánh, "vs"), `long_term` (dùng lâu).

**Lưu ý:** loại video đoán bằng từ khóa tiêu đề, chỉ để thống kê. `long_term` hiếm vì crawler chỉ lấy video đăng trong cửa sổ ra mắt.

In [ ]:
if vd is not None:
    theo_loai = (bl_vd.assign(video_type=bl_vd["video_type"].fillna("(không rõ)")).groupby("video_type")
                   .agg(**{"video": ("video_id", "nunique"), "bình luận": ("comment_id", "size")})
                   .sort_values("bình luận", ascending=False))
    theo_loai["% bình luận"] = (100 * theo_loai["bình luận"] / len(df)).round(2)
    theo_loai["bình luận / video"] = (theo_loai["bình luận"] / theo_loai["video"]).round(1)
    for loai, dong in theo_loai.iterrows():
        KQ[f"% bình luận dưới video loại {loai}"] = dong["% bình luận"]
    display(theo_loai)

### 4.3 Video tặng quà

**Để làm gì:** video tặng quà, minigame kéo về rất nhiều bình luận kiểu "em xin", "cho em một suất", không nói gì về máy. Mục 2.1 cho thấy vài video loại này nằm trong nhóm nhiều bình luận nhất.

**Ô code làm gì:**
- Đánh dấu video có tiêu đề chứa `QUA_RE` (tặng, giveaway, quà, minigame, trúng thưởng, bốc thăm…).
- In số video, số bình luận dưới các video đó, danh sách video, và ví dụ bình luận ngẫu nhiên.

**Đọc kết quả:** có loại các bình luận này khỏi phân tích hay không là quyết định của nhóm, ghi ở kế hoạch nhánh `docs/0-chung/ke-hoach.md`, mục quyết định đang chờ. Notebook chỉ đếm. Từ khóa có thể bắt nhầm (vd "món quà" trong tiêu đề review), nên xem danh sách video trước khi tin con số.

In [ ]:
QUA_RE = r"tặng|giveaway|give away|quà|trúng thưởng|minigame|mini game|bốc thăm|tang qua|trung thuong"
if vd is not None:
    tieu_de = vd["title"].fillna("").map(lambda s: unicodedata.normalize("NFC", s).lower())
    video_qua = vd.index[tieu_de.str.contains(QUA_RE, regex=True)]
    la_bl_qua = df["video_id"].isin(video_qua)
    KQ["Video có tiêu đề tặng quà"] = len(video_qua)
    KQ["Bình luận dưới video tặng quà"] = int(la_bl_qua.sum())
    KQ["% bình luận dưới video tặng quà"] = round(100 * la_bl_qua.mean(), 2)
    print(f"Video tặng quà: {len(video_qua):,} | bình luận dưới các video đó: {la_bl_qua.sum():,} ({100 * la_bl_qua.mean():.2f}%)")
    top_qua = (df[la_bl_qua].groupby("video_id").size().sort_values(ascending=False).rename("bình luận").to_frame()
                 .join(vd[["channel_title", "title"]]))
    display(top_qua.head(N_SHOW))
    if la_bl_qua.any():
        vi_du_qua = df.loc[la_bl_qua, ["model", "text"]].sample(min(20, int(la_bl_qua.sum())), random_state=0)
        print("Ví dụ bình luận dưới video tặng quà:")
        display(vi_du_qua.assign(text=vi_du_qua["text"].fillna("").str.replace("\n", " ⏎ ", regex=False).str.slice(0, 200)))

### 4.4 Reply và chủ kênh

**Ô code làm gì:**
- Tỷ lệ bình luận là reply (`parent_id` khác rỗng). YouTube chỉ có 2 tầng, mọi reply trỏ về bình luận gốc.
- Số reply đã crawl trên mỗi bình luận gốc; % bình luận gốc có ít nhất một reply.
- So tổng `reply_count` (YouTube báo cho bình luận gốc) với số reply đã crawl. Chênh lệch là reply đăng ngoài cửa sổ 4 tuần (crawler không lấy) hoặc đã bị xóa.
- Reply "mồ côi": bình luận gốc của nó không có trong dữ liệu (thường do bình luận gốc đăng trước cửa sổ).
- Bình luận của chính chủ kênh (`author_is_creator`), chia theo bình luận gốc và reply.

**Đọc kết quả:** reply hay là câu trả lời ngắn ("đúng rồi", "không"), muốn hiểu phải đọc kèm bình luận gốc (`parent_text`). Bước gán nhãn và mô hình cần tính tới điều này.

In [ ]:
la_reply = df["parent_id"].notna()
goc_ids = df.loc[~la_reply, "comment_id"]
reply_moi_goc = df.loc[la_reply, "parent_id"].value_counts().reindex(goc_ids, fill_value=0)
reply_api = pd.to_numeric(df.loc[~la_reply, "reply_count"], errors="coerce").fillna(0)
mo_coi = ~df.loc[la_reply, "parent_id"].isin(set(goc_ids))
chu_kenh = df["author_is_creator"].eq(True)

KQ["% bình luận là reply"] = round(100 * la_reply.mean(), 2)
KQ["% bình luận gốc có ≥ 1 reply đã crawl"] = round(100 * (reply_moi_goc > 0).mean(), 2)
KQ["Tổng reply_count theo YouTube / reply đã crawl"] = f"{int(reply_api.sum()):,} / {int(la_reply.sum()):,}"
KQ["Reply mồ côi"] = int(mo_coi.sum())
KQ["% bình luận của chủ kênh"] = round(100 * chu_kenh.mean(), 2)
print(f"Reply: {la_reply.sum():,} ({100 * la_reply.mean():.2f}%) | bình luận gốc: {(~la_reply).sum():,}")
print(f"reply_count theo YouTube: {int(reply_api.sum()):,} | reply đã crawl: {la_reply.sum():,} | reply mồ côi: {mo_coi.sum():,}")
display(reply_moi_goc.describe(percentiles=[.5, .75, .9, .99]).round(2).to_frame("reply đã crawl / bình luận gốc"))
display(pd.crosstab(la_reply.map({True: "reply", False: "bình luận gốc"}),
                    chu_kenh.map({True: "chủ kênh", False: "người xem"}), margins=True, margins_name="tổng"))

### 4.5 Like

**Ô code làm gì:** phân bố số like mỗi bình luận, % bình luận không có like nào, và `N_SHOW` bình luận nhiều like nhất.

**Đọc kết quả:** like cho biết bao nhiêu người đồng tình với một ý. Bước sau có thể dùng like làm trọng số (một ý được 500 like nặng hơn một ý 0 like); nhóm cần quyết có dùng không.

In [ ]:
like = pd.to_numeric(df["like_count"], errors="coerce").fillna(0).astype(int)
KQ["% bình luận 0 like"] = round(100 * (like == 0).mean(), 1)
KQ["Like nhiều nhất của một bình luận"] = int(like.max())
print(f"Bình luận 0 like: {100 * (like == 0).mean():.1f}%")
display(like.describe(percentiles=[.5, .75, .9, .99]).round(1).to_frame("like / bình luận"))
top_like = df.assign(like=like).nlargest(N_SHOW, "like")[["like", "model", "video_id", "text"]]
display(top_like.assign(text=top_like["text"].fillna("").str.replace("\n", " ⏎ ", regex=False).str.slice(0, 200)))

## 5. Tổng hợp số liệu mục 3–4

**Đọc kết quả:** các con số chính của mục 3–4 trên một bảng, để chép vào báo cáo `docs/1-thu-thap/bao-cao-kiem-tra-du-lieu.md`. Số của Level 0 ở cuối mục 2.

In [ ]:
display(pd.Series(KQ, name="giá trị").astype(str).to_frame())